In [1]:
import torch
print("device_count:", torch.cuda.device_count())
print("is_available:", torch.cuda.is_available())

device_count: 1
is_available: True


In [2]:
import sys, site

user_site = site.getusersitepackages()
print("Using python:", sys.executable)
print("User site on path:", user_site in sys.path)

Using python: /opt/conda/envs/cse234/bin/python3
User site on path: True


In [3]:
import sys, site

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
from datasets import Dataset
import itertools
import json

print("RFLoraConfig:", RFLoraConfig)


RFLoraConfig: <class 'rapidfireai.automl.model_config.RFLoraConfig'>


In [4]:
import sys, site
sys.path.insert(0, site.getusersitepackages())
from peft import LoraConfig
print("peft works:", LoraConfig)

peft works: <class 'peft.tuners.lora.config.LoraConfig'>


#### Define Model Creation Function for All Model Types Across Configs

In [5]:
# from rapidfireai import Experiment
# from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
# from datasets import Dataset
# import itertools
# import json

In [6]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 512

### Load Dataset and Specify Train and Eval Partitions

In [ ]:
import json
from datasets import Dataset

### Warning This code was run before augmentating the train.json ###
### train.json is now train_original.json and augmented data is now train.json. Change before running. ###
with open("train.json", "r") as f:
    train_dataset = Dataset.from_list(json.load(f))

with open("validation.json", "r") as f:
    validation_dataset = Dataset.from_list(json.load(f))

# print(len(train_dataset), len(validation_dataset))


### Define Data Processing Function

In [ ]:

# def basic_formatting_function(row):
#     import json

#     clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
#     filepath = f"./schemas/{clean_id}.json"
#     with open(filepath) as f:
#         schema_data = json.load(f)

#     schema = {}
#     for table_name in schema_data["table_names_original"]:
#         schema[table_name] = []
#     for i, name in schema_data["column_names_original"]:
#         if i == -1:
#             continue
#         table_name = schema_data["table_names_original"][i]
#         schema[table_name].append(name)

#     system_prompt = (
#         "You are a schema-linking assistant. "
#         "Given a question and a database schema, return ONLY a valid JSON object "
#         "that maps table names to relevant column-name lists."
#     )
#     prompt = (
#         f"Database schema: {schema}\n\n"
#         f"Question: {row['question']}\n\n"
#         "Return JSON only in this format: {\"TableName\": [\"col1\", \"col2\"], ...}"
#     )
#     answer = json.dumps(row["schema_links"], ensure_ascii=False)
#     return {
#     "text": (
#         f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
#         f"<|im_start|>user\n{prompt}<|im_end|>\n"
#         f"<|im_start|>assistant\n{answer}<|im_end|>"
#     )
# }

def basic_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        table_name = schema_data["table_names_original"][i]
        schema[table_name].append(name)
    
    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema: {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return a JSON object with only the relevant tables as keys and lists of relevant column names as values. "
        "You MUST include specific column names — do not return empty lists unless a table has no relevant columns. "
        "Example: {\"Orders\": [\"order_id\", \"total\"], \"Customers\": [\"name\"]}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
        "text": (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n{answer}<|im_end|>"
        )
    }

def pkfk_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    column_info = schema_data["column_names_original"]
    primary_keys = schema_data.get("primary_keys", [])
    foreign_keys = schema_data.get("foreign_keys", [])
    col_annotations = {}

    for pk in primary_keys:
        if isinstance(pk, list):
            for col_idx in pk:
                col_annotations[col_idx] = "(PK)"
        else:
            col_annotations[pk] = "(PK)"

    for from_idx, to_idx in foreign_keys:
        to_table_idx = column_info[to_idx][0]
        to_table_name = schema_data["table_names_original"][to_table_idx]
        if from_idx in col_annotations and col_annotations[from_idx] == "(PK)":
            col_annotations[from_idx] = f"(PK,FK→{to_table_name})"
        else:
            col_annotations[from_idx] = f"(FK→{to_table_name})" 

    schema = {}
    for col_idx, (table_idx, col_name) in enumerate(column_info):
        if table_idx == -1:
            continue
        table_name = schema_data["table_names_original"][table_idx]
        if table_name not in schema:
            schema[table_name] = []
        annotation = col_annotations.get(col_idx, "")
        if annotation:
            annotated_col = f"{col_name} {annotation}"
        else:
            annotated_col = col_name
        schema[table_name].append(annotated_col)
    # used coding agent to improve prompt
    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema with PK/FK annotations, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists (without annotations in the output)."
    )
    # used coding agent to improve prompt
    prompt = (
        f"Database schema (PK/FK annotated): {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only — column names without annotations: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
    "text": (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
}
def sorted_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        table_name = schema_data["table_names_original"][i]
        schema[table_name].append(name)

    sorted_schema = {}
    for table_name in sorted(schema.keys()):
        sorted_schema[table_name] = sorted(schema[table_name])

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema (sorted): {sorted_schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only in this format: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
    "text": (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
}

### Initialize Experiment

In [ ]:
# Every experiment instance must be uniquely named
experiment = Experiment(experiment_name="experkrj_16", mode="fit")

The previously running experiment experkrj_39 was forcibly ended. Created a new experiment 'experkrj_40' with Experiment ID: 41 and Metric Experiment ID: experkrj_40 at /home/sjrao/rapidfireai/rapidfire_experiments/experkrj_40


### Define Multi-Config Knobs for Model, LoRA, and SFT Trainer using RapidFire AI Wrapper APIs

In [10]:
import torch

QWEN = "Qwen/Qwen2.5-1.5B-Instruct"
ATTN_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
ALL_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

configs_spec = [
    # --- Group A: prompt format ---
    ("A1_basic_r8",       QWEN, basic_formatting_function,  8,  16, 1e-4, 120, ALL_MODULES),
    ("A2_pkfk_r8",        QWEN, pkfk_formatting_function,   8,  16, 1e-4, 120, ALL_MODULES),
    ("A3_sorted_r8",      QWEN, sorted_formatting_function,  8,  16, 1e-4, 120, ALL_MODULES),
    # --- Group B: LoRA rank ---
    ("B1_basic_r16",      QWEN, basic_formatting_function,  16,  32, 1e-4, 120, ALL_MODULES),
    ("B2_basic_r32",      QWEN, basic_formatting_function,  32,  64, 1e-4, 120, ALL_MODULES),
    # --- Group C: learning rate ---
    ("C1_basic_lr2e4",    QWEN, basic_formatting_function,   8,  16, 2e-4, 120, ALL_MODULES),
    ("C2_basic_lr5e5",    QWEN, basic_formatting_function,   8,  16, 5e-5, 120, ALL_MODULES),
    # --- Group D: target modules ---
    ("D1_basic_attn",     QWEN, basic_formatting_function,  16,  32, 1e-4, 120, ATTN_MODULES),
    # --- Group E: combined best at 150 steps ---
    ("E1_pkfk_r16_150",   QWEN, pkfk_formatting_function,  16,  32, 1e-4, 150, ALL_MODULES),
    # --- Group F: extend top format/rank combos to 150 steps ---
    ("F1_pkfk_r8_150",    QWEN, pkfk_formatting_function,   8,  16, 1e-4, 150, ALL_MODULES),
    ("F2_basic_r16_150",  QWEN, basic_formatting_function,  16,  32, 1e-4, 150, ALL_MODULES),
    # --- Group G: original exp37 config (r=4, attn-only, lr=2e-5) at 120 steps ---
    ("G1_attn_r4_lr2e5",  QWEN, basic_formatting_function,   4,   8, 2e-5, 120, ATTN_MODULES),
    # --- Group H: 200-step probe to characterise overfitting boundary ---
    ("H1_basic_r8_200",   QWEN, basic_formatting_function,   8,  16, 1e-4, 200, ALL_MODULES),
    ("H2_pkfk_r16_200",   QWEN, pkfk_formatting_function,  16,  32, 1e-4, 200, ALL_MODULES),
]

all_configs = []
for label, model_name, fmt_func, r, alpha, lr, max_steps, target_modules in configs_spec:
    model_kwargs = {
        "torch_dtype": torch.float16,
        "use_cache": False,
        "local_files_only": True,
    }
    all_configs.append(RFModelConfig(
        model_name=model_name,
        peft_config=RFLoraConfig(
            r=r,
            lora_alpha=alpha,
            lora_dropout=0.1,
            target_modules=target_modules,
            bias="none",
        ),
        training_args=RFSFTConfig(
            learning_rate=lr,
            lr_scheduler_type="linear",
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            max_steps=max_steps,
            gradient_accumulation_steps=4,
            gradient_checkpointing=False,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=20,
            bf16=False,
            fp16=True,
        ),
        model_type="causal_lm",
        model_kwargs=model_kwargs,
        formatting_func=fmt_func,
    ))

print(f"Total configs: {len(all_configs)}")
for i, (label, *_) in enumerate(configs_spec):
    print(f"  [{i+1}] {label}")
config_set = List(all_configs)

Total configs: 14
  [1] A1_basic_r8
  [2] A2_pkfk_r8
  [3] A3_sorted_r8
  [4] B1_basic_r16
  [5] B2_basic_r32
  [6] C1_basic_lr2e4
  [7] C2_basic_lr5e5
  [8] D1_basic_attn
  [9] E1_pkfk_r16_150
  [10] F1_pkfk_r8_150
  [11] F2_basic_r16_150
  [12] G1_attn_r4_lr2e5
  [13] H1_basic_r8_200
  [14] H2_pkfk_r16_200


#### Define Model Creation Function for All Model Types Across Configs

In [11]:
def sample_create_model(model_config):
    """Creates model + tokenizer with conservative CUDA settings for worker stability."""
    import gc
    import os
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["HF_DATASETS_OFFLINE"] = "1"

    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_name = model_config["model_name"]
    model_kwargs = dict(model_config["model_kwargs"])
    model_kwargs.pop("device_map", None)
    model_kwargs.setdefault("low_cpu_mem_usage", True)

    use_4bit = model_kwargs.pop("use_4bit", False)
    if use_4bit:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        model_kwargs["quantization_config"] = bnb_config

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
    return (model, tokenizer)


#### Generate Config Group

In [12]:
config_group = RFGridSearch(
    configs=config_set,
    trainer_type="SFT"
)

### Run Multi-Config Training

In [13]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,       
    validation_dataset,
    num_chunks=1,
    seed=42,
)


INFO 06-01 01:14:33 [__init__.py:216] Automatically detected platform cuda.


/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/__init__.py:22: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_client.py:40: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_generation.py:41: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


Started 1 worker processes successfully
Created workers


/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/__init__.py:22: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():
/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_client.py:40: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


/home/sjrao/.local/lib/python3.12/site-packages/trl/generation/vllm_generation.py:41: UserWarning: TRL currently supports vLLM versions from 0.12.0 to 0.18.0. You have version 0.10.2 installed. We recommend installing a supported version to avoid compatibility issues.
  if is_vllm_available():


### End Current Experiment

In [14]:
experiment.end()
# Experiment(experiment_name="experkrj_16", mode="fit").end()

Experiment experkrj_40 ended
Workers stopped
